# One large prompt versus multi-step prompting

This notebook plans the same trip in two ways. The first asks for everything in one request. The second breaks the work into requirements, a daily outline, and a final itinerary.

In [ ]:
from os import getenv

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()


def groq_completion(messages: list[dict[str, str]], max_tokens: int = 1000) -> str:
    api_key = getenv("GROQ_API_KEY")
    if not api_key:
        raise ValueError("Set GROQ_API_KEY in your environment or .env file.")

    client = OpenAI(
        api_key=api_key,
        base_url="https://api.groq.com/openai/v1",
    )
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        temperature=0,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


In [3]:
trip_brief = """
Plan a four-day trip for two adults. They arrive in Kyoto and depart from Osaka in mid-November.
They prefer vegetarian-friendly food, moderate walking, and no more than four activities each day.
Include transit suggestions, but do not claim live schedules, prices, or availability.
"""


## One large prompt

This approach places every requirement and the requested deliverable in one message. It is concise, but gives us no opportunity to inspect or correct the model's interpretation before it writes the full itinerary.

In [5]:
one_big_prompt = f"""
{trip_brief}

Create a complete day-by-day itinerary. For each day, include a morning, afternoon, evening,
one vegetarian-friendly meal suggestion, and the most practical way to travel between areas.
Explain briefly why the sequence makes sense.
"""

one_big_plan = groq_completion([
    {"role": "system", "content": "You are a careful trip-planning assistant. Do not present estimates as live information."},
    {"role": "user", "content": one_big_prompt},
])

print(one_big_plan)


**Four‑Day Kyoto‑to‑Osaka Itinerary (mid‑November)**  
*Two adults – vegetarian‑friendly meals, moderate walking, ≤ 4 activities per day*  

---

## Day 1 – Arrival & First Glimpse of Kyoto  
**Morning – Arrival in Kyoto**  
*Activity*: Drop luggage at your hotel/ryokan in central Kyoto (e.g., near Kawaramachi or Gion).  
*Why*: Staying in the historic centre puts the main sights within easy walking distance and gives quick access to public transport for later days.  

**Afternoon – Kinkaku‑ji (Golden Pavilion) & Ryoan‑ji**  
*Transit*: Take the city bus from Kawaramachi (or the nearest subway stop) to Kinkaku‑ji; then a short bus ride to Ryoan‑ji (≈15 min total).  
*Activity*: Visit the iconic Golden Pavilion, then stroll the tranquil rock garden at Ryoan‑ji. Both sites involve short, level walks and plenty of benches for rest.  

**Evening – Gion District Walk & Dinner**  
*Transit*: Return by bus or subway to Gion (≈20 min).  
*Meal*: **Vegetarian kappō** at **Mumokuteki Café** (nea

## Multi-step prompting

This version keeps the work in one prompt and one API call, but gives the model an explicit order of operations and a required output structure.

In [4]:
multi_step_prompt = f"""
{trip_brief}

Work through this trip plan in the following order:
1. Extract the traveller requirements as concise bullet points.
2. Create a logical four-day outline, with one concise bullet for each day.
3. Expand the outline into a practical itinerary. For each day include morning, afternoon, evening,
   one vegetarian-friendly meal suggestion, and transit guidance.

Do not claim live schedules, prices, or availability. Format your response with exactly these headings:
## Requirements
## Daily outline
## Final itinerary
"""

multi_step_plan = groq_completion([
    {"role": "system", "content": "You are a careful trip-planning assistant. Follow each requested step in order."},
    {"role": "user", "content": multi_step_prompt},
], max_tokens=1_000)

print(multi_step_plan)


## Requirements
- Arrival: Kyoto (mid‑November)  
- Departure: Osaka (mid‑November)  
- Travelers: 2 adults  
- Food: Vegetarian‑friendly options preferred  
- Activity level: Moderate walking, no more than 4 activities per day  
- Transit: Include suggestions for trains, buses, or subways (no live schedules, prices, or availability)  

## Daily outline
- **Day 1 – Kyoto:** Explore historic central Kyoto, visit a temple, stroll a garden, enjoy a vegetarian dinner in Gion.  
- **Day 2 – Kyoto outskirts:** Day trip to Arashiyama bamboo grove and nearby attractions, return to central Kyoto for evening.  
- **Day 3 – Transfer to Osaka:** Morning in Kyoto, then travel to Osaka; afternoon in Osaka Castle Park, evening in Dotonbori.  
- **Day 4 – Osaka:** Visit a museum, shop in Shinsaibashi, relax in a park, depart from Osaka station.  

## Final itinerary

### Day 1 – Kyoto (Arrival)
**Morning**  
- Arrive in Kyoto and check into accommodation near Kyoto Station.  
- Walk to **Kyoto Tower**

## Compare the approaches

Both versions use one API call. The multi-step version makes the intermediate reasoning visible through required sections, which makes the final itinerary easier to inspect and revise. For separate review points between calls, use prompt chaining instead.